<a href="https://colab.research.google.com/github/benameur21/Brain-Tumor-Segmentation-UNet-TL/blob/main/Brain_tumor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/benameur21/Brain-Tumor-Segmentation-UNet-TL.git

Cloning into 'Brain-Tumor-Segmentation-UNet-TL'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [3]:
%cd Brain-Tumor-Segmentation-UNet-TL.git
!git status

[Errno 2] No such file or directory: 'Brain-Tumor-Segmentation-UNet-TL.git'
/content
fatal: not a git repository (or any of the parent directories): .git


# Part A

**A0.Environment Setup**

Install Required Libraries

In [5]:
!pip install -q segmentation-models-pytorch==0.3.4
!pip install -q albumentations
!pip install -q opencv-python matplotlib pandas scikit-learn

Import Libraries

In [6]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp

Set Random Seed

In [7]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


Detect Computing Device

In [8]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Using CPU.")

Selected device: cuda
GPU: Tesla T4


Q01 : The chosen hardware is the GPU (using CUDA) specifically an NVIDIA Tesla T4 it enhances the speed of simultaneous calculations leading to a decrease in training duration when set against the CPU

**A1.LGG MRI Dataset Exploration**

Dataset Inventory

*Define Dataset Path and List Files*

In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mateuszbuda/lgg-mri-segmentation")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'lgg-mri-segmentation' dataset.
Path to dataset files: /kaggle/input/lgg-mri-segmentation


In [12]:
import os
import glob

DATASET_DIR = "/kaggle/input/lgg-mri-segmentation"

all_files = glob.glob(
    os.path.join(DATASET_DIR, "**", "*.tif"),
    recursive=True
)

print("Dataset directory:", DATASET_DIR)
print("Total TIFF files:", len(all_files))
print("First 5 files:", all_files[:5])

Dataset directory: /kaggle/input/lgg-mri-segmentation
Total TIFF files: 15716
First 5 files: ['/kaggle/input/lgg-mri-segmentation/kaggle_3m/TCGA_DU_7010_19860307/TCGA_DU_7010_19860307_45.tif', '/kaggle/input/lgg-mri-segmentation/kaggle_3m/TCGA_DU_7010_19860307/TCGA_DU_7010_19860307_56_mask.tif', '/kaggle/input/lgg-mri-segmentation/kaggle_3m/TCGA_DU_7010_19860307/TCGA_DU_7010_19860307_57.tif', '/kaggle/input/lgg-mri-segmentation/kaggle_3m/TCGA_DU_7010_19860307/TCGA_DU_7010_19860307_33.tif', '/kaggle/input/lgg-mri-segmentation/kaggle_3m/TCGA_DU_7010_19860307/TCGA_DU_7010_19860307_27.tif']


*Separate Images and Masks*

In [13]:
mask_paths = [
    path for path in all_files
    if path.endswith("_mask.tif")
]

mri_paths = [
    path for path in all_files
    if not path.endswith("_mask.tif")
]

print("MRI images:", len(mri_paths))
print("Segmentation masks:", len(mask_paths))

MRI images: 7858
Segmentation masks: 7858


*Match Images and Masks*

In [14]:
from pathlib import Path

mask_dict = {
    Path(path).stem.replace("_mask", ""): path
    for path in mask_paths
}

image_mask_pairs = []

for image_path in mri_paths:
    image_name = Path(image_path).stem

    if image_name in mask_dict:
        image_mask_pairs.append(
            (image_path, mask_dict[image_name])
        )

print("Matched image-mask pairs:", len(image_mask_pairs))

Matched image-mask pairs: 7858


*Count Patients*

In [18]:
patient_ids = {
    "_".join(Path(image_path).stem.split("_")[:3])
    for image_path, mask_path in image_mask_pairs
}

print("Number of patients:", len(patient_ids))
print("First 10 patient IDs:", sorted(patient_ids)[:10])

Number of patients: 110
First 10 patient IDs: ['TCGA_CS_4941', 'TCGA_CS_4942', 'TCGA_CS_4943', 'TCGA_CS_4944', 'TCGA_CS_5393', 'TCGA_CS_5395', 'TCGA_CS_5396', 'TCGA_CS_5397', 'TCGA_CS_6186', 'TCGA_CS_6188']
